# 15 — SQLite, SQL, and Transactions

SQLite is a lightweight embedded SQL database included with Python. Practice schema design, parameterized queries, CRUD, constraints, and transactions without creating persistent files.

*Intermediate · 75–120 min*

## You will learn

- Create tables with constraints and stable keys
- Insert, query, update, and delete records
- Use parameterized SQL to avoid injection
- Understand transaction commit, rollback, and connection cleanup

## Examples

### 1. Connecting and creating tables

sqlite3.connect(':memory:') makes an in-memory database that disappears when the connection is closed. CREATE TABLE defines columns and constraints. Always parameterize external values rather than interpolating them into SQL text.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
try:
    connection.execute("""
        CREATE TABLE students (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL UNIQUE,
            score REAL NOT NULL CHECK(score BETWEEN 0 AND 100)
        )
    """)
    connection.executemany(
        "INSERT INTO students (name, score) VALUES (?, ?)",
        [("Ada", 91), ("Ben", 78), ("Cam", 85)]
    )
    print(connection.execute("SELECT name, score FROM students ORDER BY score DESC").fetchall())
finally:
    connection.close()

**Think about it:** Why does the CHECK constraint protect against an invalid score?

### 2. CRUD with bound parameters

CRUD means Create, Read, Update, Delete. Placeholder parameters keep query structure separate from data and prevent values from being interpreted as SQL syntax. SELECT can include filters, ordering, and limits.

In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as db:
    db.execute("CREATE TABLE products (sku TEXT PRIMARY KEY, price REAL)")
    db.executemany("INSERT INTO products VALUES (?, ?)", [("P1", 10), ("P2", 20)])
    db.execute("UPDATE products SET price = ? WHERE sku = ?", (12.5, "P1"))
    print(db.execute("SELECT * FROM products WHERE price >= ? ORDER BY sku", (10,)).fetchall())
    db.execute("DELETE FROM products WHERE sku = ?", ("P2",))
    print(db.execute("SELECT COUNT(*) FROM products").fetchone()[0])

**Think about it:** What is the difference between a value parameter and a table identifier in SQL?

### 3. Transactions and rollback

A transaction groups changes into an atomic unit. Successful operations can commit together; an error should roll back the entire unit when partial changes would be harmful. The connection context manager manages commit/rollback but does not itself close the connection.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER CHECK(balance >= 0))")
    db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 40)])
    db.commit()
    try:
        with db:
            db.execute("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
    except sqlite3.IntegrityError:
        print("Transfer rejected; transaction rolled back")
    print(db.execute("SELECT * FROM accounts ORDER BY id").fetchall())
finally:
    db.close()

**Think about it:** Why would a two-step transfer require both changes in the same transaction?

### 4. Aggregations and joins

SQL databases can summarize data using COUNT, SUM, AVG, GROUP BY, and join related tables. A foreign key relates records from different tables. In SQLite, enable foreign-key enforcement per connection when you rely on it.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("PRAGMA foreign_keys = ON")
    db.execute("CREATE TABLE categories (id INTEGER PRIMARY KEY, name TEXT)")
    db.execute("CREATE TABLE sales (category_id INTEGER REFERENCES categories(id), amount REAL)")
    db.executemany("INSERT INTO categories VALUES (?, ?)", [(1, "books"), (2, "pens")])
    db.executemany("INSERT INTO sales VALUES (?, ?)", [(1, 10), (1, 15), (2, 3)])
    query = """SELECT c.name, COUNT(*), SUM(s.amount)
               FROM sales s JOIN categories c ON s.category_id = c.id
               GROUP BY c.id, c.name ORDER BY c.name"""
    print(db.execute(query).fetchall())
finally:
    db.close()

**Think about it:** What happens when a sale references a category id that does not exist with foreign keys enabled?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/04_sql_http_cli_performance_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/15', {1: '4637b24fb70f017a', 2: 'c46656e3bf62b636', 3: 'fb62c0bbcd0ea769', 4: 'ee84aaac1c8e2e84'})

**Exercise 1.** Create a tasks table with a primary key and a done flag.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
columns = [row[1] for row in conn.execute("PRAGMA ___(tasks)")]
check(1, columns)

**Exercise 2.** Insert a task safely and return its generated id.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")

def add_task(conn, title):
    cursor = conn.execute("INSERT INTO tasks (title) VALUES (___)", (title,))
    return cursor.___

check(2, [add_task(conn, "write"), add_task(conn, "test")])

**Exercise 3.** Filter tasks by status with a parameterized query.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (title TEXT, done INTEGER)")
conn.executemany("INSERT INTO tasks VALUES (?, ?)", [("a", 1), ("b", 0), ("c", 1)])
done = [t for (t,) in conn.execute("SELECT title FROM tasks WHERE done = ? ORDER BY title", (___,))]
check(3, done)

**Exercise 4.** A transaction that rolls back when a constraint is violated.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.execute("INSERT INTO accounts VALUES ('ana', 50), ('ben', 0)")
conn.commit()
try:
    with ___:                 # commits on success, rolls back on error
        conn.execute("UPDATE accounts SET balance = balance + 80 WHERE name = 'ben'")
        conn.execute("UPDATE accounts SET balance = balance - 80 WHERE name = 'ana'")
except sqlite3.___:
    pass
check(4, dict(conn.execute("SELECT name, balance FROM accounts")))

## Common mistakes

- Never construct SQL using untrusted user text as query syntax.
- The sqlite3 connection context manager does not close the connection.
- Remember PRAGMA foreign_keys = ON if you depend on foreign-key checks.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why write `execute('… WHERE id = ?', (task_id,))`?

   - **a)** Faster typing
   - **b)** SQLite requires it
   - **c)** Prevents SQL injection

**2.** `with conn:` in sqlite3…

   - **a)** Closes the connection
   - **b)** Opens a new database
   - **c)** Commits on success, rolls back on error

**3.** A CHECK constraint is enforced by…

   - **a)** Nobody
   - **b)** Your Python code
   - **c)** The database itself

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/15', {1: '95bcf96ede2a', 2: '7a3655181fa2', 3: '4531163d339b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [16 — HTTP, JSON APIs, and Reliable Clients](16_http_apis_and_clients.ipynb)